# Target Regions for a Parkinson’s Disease Virtual Gene Panel


## Overview


1.  Query panels in `PanelApp API`.
2.  Clean, merge, and filter GREEN (diagnostic quality) entities.
3.  Query GREEN gene data in `NCBI Datasets API`.
4.  Clean recovered gene data.
5.  Enrich Panel with RefSeq annotations.
6.  Create BED file with exons, STRs, CNVs.


In [ ]:
# Imports and setup

%load_ext autoreload
%autoreload 2

import logging
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib_venn import venn2 #TODO: add this package to requirements.txt


logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
project_root = Path().resolve().parent

# Import custom source code
sys.path.append(str(project_root / "src"))
from virtual_panel import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Retrieve & Structure Panels


In [ ]:
# Query PanelApp APIs

panelapp_queries: list[PanelAppPanelId] = [
    # Parkinson Disease and Complex Parkinsonism (Version 1.127), Genomics England PanelApp
    PanelAppPanelId(
        healthcare_system=PanelAppHealthSystem.ENGLAND, id="39", version="1.127"
    ),
    # Early-onset Parkinson disease (Version 2.36), PanelApp Australia
    PanelAppPanelId(
        healthcare_system=PanelAppHealthSystem.AUSTRALIA, id="26", version="2.36"
    ),
]

results: dict[str, Any] = dict()
for query in panelapp_queries:
    logging.info(f"Querying {query}...")
    response = PanelAppClient(query.healthcare_system).get_single_panel(query)
    results[f"{query}"] = response
    logging.info("Done!\n")

In [ ]:
# Structure PanelApp API response and prepare for further cleaning

panelapp_panels: dict[str, PanelAppPanel] = dict()
for query, raw_panel in results.items():
    logging.info(f"Parsing response for {query}...")
    panelapp_panels[query] = PanelAppPanel.from_raw_panel(raw_panel)
    logging.info(f"Done!\n")

In [ ]:
# Check Panels' Metadata and entities' status

for query, panel in panelapp_panels.items():
    print(f"{query}\n{panel.metadata}", flush=True)
    display(pd.crosstab(panel.df["Type"], panel.df["Status"], margins=True))

## Consensus Panels' GREEN (Diagnostic Quality) Entities


In [ ]:
# Pair to merge
keys_to_merge = ("PA-AUSTRALIA_26_v2.36", "PA-ENGLAND_39_v1.127")
suffixes_map = {"PA-AUSTRALIA_26_v2.36": "_AUS", "PA-ENGLAND_39_v1.127": "_UK"}

# Visual characteristics

venn2(
    (
        set(panelapp_panels[keys_to_merge[0]].df["Name"]),
        set(panelapp_panels[keys_to_merge[1]].df["Name"]),
    ),
    (
        panelapp_panels[keys_to_merge[0]].metadata.name,
        panelapp_panels[keys_to_merge[1]].metadata.name,
    ),
)

In [ ]:
# Merge and filter GREEN entities

panelapp_merged = PanelAppMerged.new(
    panelapp_panels[keys_to_merge[0]],
    panelapp_panels[keys_to_merge[1]],
    keys_to_merge[0],
    keys_to_merge[1],
    suffixes_map[keys_to_merge[0]],
    suffixes_map[keys_to_merge[1]],
)

panelapp_merged.df["_merge"].value_counts(dropna=False)

In [ ]:
# List conflicts that may require manual resolution

for col_name, conflict in panelapp_merged.conflicts.items():
    print(f"Conflict details for column {col_name}:", flush=True)

    conflict_col1 = f"{col_name}{panelapp_merged.suffix_left}"
    conflict_col2 = f"{col_name}{panelapp_merged.suffix_right}"

    display(
        pd.crosstab(
            conflict[conflict_col1],
            conflict[conflict_col2],
            margins=True,
        )
    )

    display(
        conflict[
            [
                conflict_col1,
                conflict_col2,
            ]
        ]
    )

In [ ]:
# TODO: Explain complexities of panel harmonization
# TODO: Since Status conflict is expected for most of the cases, may be worth to add this as a default solving strategy

conflicted_field = "Status"
consensus_col_name = f"{conflicted_field}_Consensus"

panelapp_merged.df[consensus_col_name] = panelapp_merged.df.apply(
    lambda row, col_name_left, col_name_right: (
        "GREEN"
        if row[col_name_left]
        == row[col_name_right]
        else "MIXED"
    ), args = [f"{conflicted_field}{panelapp_merged.suffix_left}", f"{conflicted_field}{panelapp_merged.suffix_right}"]
    axis=1,
)

panelapp_merged.df[consensus_col_name] = panelapp_merged.df[consensus_col_name].astype(
    "category"
)

panelapp_merged.df[consensus_col_name].value_counts(dropna=False)

In [ ]:
# Build consensus DataFrame

consensus_panel_df = panelapp_merged.make_consensus(custom_include=["Status_Consensus"])

sns.catplot(data=panelapp_merged.df, x="_merge", hue="Type", kind="count")

In [ ]:
pd.crosstab(
    consensus_panel_df["Status_Consensus"],
    consensus_panel_df["Type"],
    margins=True,
)

In [ ]:
# TODO: make chromosome categorical and ordered, include MT and Unplaced
sns.countplot(data=consensus_panel_df, x="GRCh38_chr")

## Retrieve & Parse Gene RefSeq Annotations


Next steps...

- Replace Ensembl 90/10X gene annotations by RefSeq
- Create BED with GENE entities exons, STRs, and CNVs regions.
- Analysis and conclusions
